In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

%matplotlib inline
# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.insert(0, project_root)

# Import psyduck framework
from psyduck import Spin
from psyduck.hamiltonians import Hz_order
from psyduck.operations import global_rotation
from psyduck.plotting import wigner_plot_hammer
from psyduck.plotting.wigner_plot import wigner_plot_3d

# Import local helper functions
import ClassicalSimFunc as cf

print("✓ All imports successful!")
print(f"QuTiP version: {qt.__version__}")


In [ ]:
# Define spin quantum number
I = 3.5  # Spin-7/2 nucleus
dim = int(2 * I + 1)  # Hilbert space dimension = 8

# Create spin system using psyduck
nucleus = Spin(I=I)

# Get spin operators
Ix, Iy, Iz = nucleus.get_spin_operators()

print(f"✓ Initialized spin system")
print(f"  Quantum number: I = {I}")
print(f"  Hilbert space dimension: {dim}")
print(f"  Spin operators shape: {Ix.shape}")

In [ ]:
# Simulation parameters
tau = 1.0              # Free evolution time between kicks
kappa = 3.0            # Kick strength (enters chaotic regime)
order = 2              # Nonlinearity order: p=2 (quadratic)
n_kicks = 20           # Number of kicks to simulate

# Build the Floquet operator for a single period: U = U_kick @ U_free
# Free evolution Hamiltonian: H0 = (π/2τ) * (-Iy)
H_free = (np.pi / 2.0) * (-Iy)
U_free = (-1j * H_free * tau).expm()

# Nonlinear kick Hamiltonian using psyduck's Hz_order
H_kick = Hz_order(kappa, order, I)
U_kick = (-1j * H_kick).expm()

# Combined Floquet operator for one period
U_floquet = U_kick @ U_free

print(f"✓ Constructed Floquet operators")
print(f"\nParameters:")
print(f"  τ (free evolution time) = {tau}")
print(f"  κ (kick strength) = {kappa:.4f}")
print(f"  p (nonlinearity order) = {order}")
print(f"  Floquet operator shape: {U_floquet.shape}")
print(f"  Unitarity check ||U†U - I|| = {np.linalg.norm((U_floquet.dag() @ U_floquet - qt.qeye(dim)).full()):.2e}")

In [ ]:
# Eigendecompose the Floquet operator
eigvals, eigvecs = U_floquet.eigenstates()

# Extract and normalize eigenvalues to unit circle
eigvals = np.array(eigvals, dtype=complex)
magnitudes = np.abs(eigvals)
eigvals_normalized = eigvals / np.where(magnitudes == 0, 1.0, magnitudes)

# Extract eigenphases
phases = np.mod(np.angle(eigvals), 2 * np.pi)
phases_sorted_idx = np.argsort(phases)
phases_sorted = phases[phases_sorted_idx]
eigvecs_sorted = [eigvecs[i] for i in phases_sorted_idx]

print(f"✓ Floquet analysis complete")
print(f"\nEigenvalue spectrum (eigenphases):")
for i, (phase, eig) in enumerate(zip(phases_sorted, eigvals_normalized[phases_sorted_idx])):
    print(f"  λ_{i} = e^(i·{phase:.4f}) rad = {eig.real:.4f} + i·{eig.imag:.4f}")

In [ ]:
# Plot eigenvalues on unit circle
fig, ax = plt.subplots(figsize=(6, 6))

# Unit circle
theta_circle = np.linspace(0, 2*np.pi, 400)
ax.plot(np.cos(theta_circle), np.sin(theta_circle), 'w--', alpha=0.3, lw=1, label='Unit circle')

# Eigenvalues
ax.scatter(
    np.real(eigvals_normalized[phases_sorted_idx]),
    np.imag(eigvals_normalized[phases_sorted_idx]),
    color="red",s=100, edgecolors='white', linewidth=1.5,
    label='Floquet eigenvalues'
)

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel('Re(λ)', fontsize=12)
ax.set_ylabel('Im(λ)', fontsize=12)
ax.set_title(f'Floquet Spectrum (κ={kappa}, p={order})', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Floquet eigenvalues visualization complete")

### 3.2 Floquet Eigenphase Distribution

**Physical Interpretation:**

The eigenphases $\{\omega_n\}$ of the Floquet unitary encode crucial dynamical information:

- **Eigenphase spacing**: $\Delta \omega_n = \omega_{n+1} - \omega_n$ determines quantum revival timescales. A regular (integrable) system has clustered phases; a chaotic system shows statistics of level repulsion (GOE/GUE statistics).

- **Symmetry sectors**: Due to rotation symmetry around the z-axis, eigenphases often cluster into groups. The separation between sectors (typically $\sim 2\pi/4$) indicates approximate degeneracies related to conserved charges.

- **Phase wrapping**: Phases are plotted in $[0, 2\pi)$, representing the quasienergy $\epsilon_n = \hbar \omega_n$ of Floquet state $n$. States with $\omega_n \approx 0$ and $\omega_n \approx 2\pi$ are nearly degenerate (periodic on the torus).

The level spacing distribution (lower right panel) is a hallmark of quantum chaos:
- **Regular systems** → Poisson statistics (exponential spacing distribution)
- **Chaotic systems** → GOE (Gaussian Orthogonal Ensemble) → level repulsion, avoiding small gaps


### 3.3 Visualize Floquet Eigenstates in Phase Space

We plot the Wigner function (or related quasi-probability distribution) for each Floquet eigenstate to visualize their structure in phase space. Scarred eigenstates will show concentration along classical orbits, while generic eigenstates display ergodic (smooth) distributions.

**Key observations:**
- **Scarred states** show sharp, localized peaks along certain directions (the classical orbit)
- **Generic states** appear smoothly distributed across the sphere (ergodic)
- **Symmetry** of each eigenstate is revealed in its Wigner pattern


## Section 4: Classical Phase Space - Poincaré Sections

To understand quantum dynamics, we analyze the underlying classical phase space by computing Poincaré sections of the classical kicked-top map.

The method used here is the **stroboscopic Poincaré map method**, also called a **stroboscopic Poincaré section** or **Poincaré surface-of-section method**. The word *stroboscopic* is important: because the kicked top is driven periodically, we do not record the classical spin continuously during the free rotation and kick. Instead, we observe the system once per driving period, immediately after each complete kick cycle. This is analogous to flashing a strobe light at the same phase of every period and recording where the spin appears.

For the kicked top, the classical state is a unit vector on the Bloch sphere,

$$\mathbf{v} = (x,y,z), \qquad |\mathbf{v}| = 1,$$

or equivalently a point in spherical coordinates $(\theta, \phi)$. One full classical kick period is represented by a discrete map

$$\mathbf{v}_{n+1} = F(\mathbf{v}_n),$$

where $F$ is implemented in `cf.kicked_top_step`. In this notebook, one step consists of two exact rotations:

1. A nonlinear torsion, or twist, about the $z$ axis:

$$\mathbf{v}' = R_z\!\left[-\kappa\,\mathrm{sgn}(z)|z|^{p-1}\right]\mathbf{v}_n.$$

2. A rigid rotation about the $y$ axis:

$$\mathbf{v}_{n+1} = R_y(-\alpha)\mathbf{v}'.$$

Here $\kappa$ is the kick strength, $p$ is the nonlinearity order, and $\alpha = \pi/2$ is the classical rotation angle corresponding to the free precession used in the quantum Floquet operator. For the common quadratic kicked top, $p=2$, the twist angle is proportional to $z$: points near the north and south poles rotate differently from points near the equator. This state-dependent rotation is what stretches and folds phase space, creating regular islands, chaotic seas, and unstable periodic structures.

The Poincaré plot is generated by **direct iteration of this discrete map**. The helper `cf.generate_poincare_section` first chooses a grid of initial conditions on the sphere: `n_seeds_phi` azimuthal values and `n_seeds_theta` polar values. Each seed is converted from $(\theta, \phi)$ into a Cartesian unit vector. Then every seed is advanced repeatedly by the same one-period map $F$.

The first `n_discard` iterates are dropped before plotting. This transient discard is a common numerical convention: early points can reflect the arbitrary choice of initial grid rather than the long-time invariant structures of the map. After the discard window, every subsequent iterate is converted back to $(\theta, \phi)$ and stored. With the parameters used below, the notebook uses

$$12 \times 12 \times (1500 - 100) = 201{,}600$$

recorded phase-space points. These points are not random samples; they are deterministic stroboscopic snapshots of many classical trajectories under the same Floquet map.

Interpreting the final scatter plot:

- Smooth closed curves indicate regular quasiperiodic motion on invariant tori.
- Dense two-dimensional clouds indicate chaotic motion, where nearby initial conditions separate and trajectories wander through an ergodic region.
- Empty gaps or island chains indicate dynamically forbidden or regular regions embedded in the chaotic sea.
- Isolated repeating structures correspond to periodic orbits or their surrounding stability islands.

So, in short: the notebook generates the Poincaré map by **stroboscopically sampling direct iterates of the classical kicked-top Floquet map**. The standard name for this construction is the **stroboscopic Poincaré section method**.

### 4.1 Visualize Poincaré Section

In [ ]:
n = 256
x = np.linspace(0, 1, n)
y = np.linspace(0, 1, n)
X, Y = np.meshgrid(x, y)

gradient = (X + Y) / 2  # diagonal: top-left → bottom-right

fig, ax = plt.subplots()
ax.imshow(gradient, cmap='nipy_spectral',
origin='lower', extent=[0, 1, 0, 1])
plt.show()


In [ ]:
# Plot: each point colored by its seed's starting position

# cmap = 'viridis'
cmap = 'nipy_spectral'

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(phi_arr.ravel(), theta_arr.ravel(),
                c=color_arr.ravel(), cmap=cmap, s=0.5, alpha=0.5)

ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(0, np.pi)
ax.set_xlabel(r'$\phi$ (azimuthal angle)', fontsize=12)
ax.set_ylabel(r'$\theta$ (polar angle)', fontsize=12)
ax.set_title(f'Poincaré Section colored by starting (θ, φ)  (κ={kappa_classical})', fontsize=13)

ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
ax.set_yticks([0, np.pi/2, np.pi])
ax.set_yticklabels([r'$0$', r'$\pi/2$', r'$\pi$'])
ax.grid(True, alpha=0.3, linestyle=':')

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Seed diagonal coordinate', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Animate the Hammer-projected Poincaré section across a sweep of κ (or α).
# Each frame re-runs the classical kicked-top map for one parameter value.

import io
from PIL import Image
from IPython.display import Image as IPyImage, display

# --- sweep configuration -------------------------------------------------
sweep_param = 'kappa'                       # 'kappa' or 'alpha'
kappa_sweep = np.linspace(0.1, 4.0, 50)     # κ values if sweep_param='kappa'
alpha_sweep = np.linspace(0.1, np.pi, 30)   # α values if sweep_param='alpha'
fixed_kappa = 3.0                           # used when sweep_param='alpha'
fixed_alpha = np.pi / 2                     # used when sweep_param='kappa'

# Lighter than the main run so the animation builds in seconds, not minutes
n_iter_anim    = 600
n_discard_anim = 50
cmap           = 'nipy_spectral'

filename = 'kicked_top_sweep.gif'
fps      = 5
dpi      = 100

sweep_vals = kappa_sweep if sweep_param == 'kappa' else alpha_sweep
# -------------------------------------------------------------------------

# Reusable seed grid (cheap, depends only on n_seeds_phi/theta)
phis_s   = np.linspace(-np.pi, np.pi, n_seeds_phi, endpoint=False)
thetas_s = np.linspace(0.15 * np.pi, 0.85 * np.pi, n_seeds_theta)

seeds_cart_a, seeds_th0_a, seeds_ph0_a = [], [], []
for th in thetas_s:
    for ph in phis_s:
        seeds_cart_a.append(cf.sph_to_cart(th, ph))
        seeds_th0_a.append(th)
        seeds_ph0_a.append(ph)
seeds_cart_a = np.stack(seeds_cart_a, axis=0)
seeds_th0_a  = np.asarray(seeds_th0_a)
seeds_ph0_a  = np.asarray(seeds_ph0_a)
n_seeds_a    = seeds_cart_a.shape[0]
c_seed_a     = (seeds_th0_a / np.pi) + ((seeds_ph0_a + np.pi) / (2 * np.pi))


def run_map(kappa_val, alpha_val):
    """Iterate the classical kicked-top map and return (phi, theta) point clouds."""
    v = seeds_cart_a.copy()
    n_keep_a = n_iter_anim - n_discard_anim
    phi_a   = np.empty((n_keep_a, n_seeds_a))
    theta_a = np.empty((n_keep_a, n_seeds_a))
    for i in range(n_iter_anim):
        v = np.array([cf.kicked_top_step(vi, kappa_val, alpha_val, order) for vi in v])
        if i >= n_discard_anim:
            ang = np.array([cf.spherical_angles(vi) for vi in v])
            phi_a[i - n_discard_anim]   = ang[:, 0]
            theta_a[i - n_discard_anim] = ang[:, 1]
    return phi_a, theta_a


frames = []
for k, val in enumerate(sweep_vals):
    if sweep_param == 'kappa':
        kappa_val, alpha_val = val, fixed_alpha
        label = fr'$\kappa$ = {val:.2f},  $\alpha$ = {alpha_val:.2f}'
    else:
        kappa_val, alpha_val = fixed_kappa, val
        label = fr'$\kappa$ = {kappa_val:.2f},  $\alpha$ = {val:.2f}'

    phi_a, theta_a = run_map(kappa_val, alpha_val)

    lon = phi_a.ravel()
    lat = np.pi / 2 - theta_a.ravel()
    col = np.broadcast_to(c_seed_a, phi_a.shape).ravel()

    fig = plt.figure(figsize=(8, 4.5))
    ax = fig.add_subplot(111, projection='hammer')
    ax.scatter(lon, lat, c=col, cmap=cmap, s=0.4, alpha=0.5,
               vmin=c_seed_a.min(), vmax=c_seed_a.max())
    ax.set_xticks([])
    ax.grid(True, alpha=0.3, linestyle=':')
    ax.set_title(f'Poincaré section (Hammer)  —  {label}',
                 fontsize=12, pad=15)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())
    plt.close(fig)
    print(f'  frame {k+1:>3}/{len(sweep_vals)}  ({label})')

frames[0].save(
    filename,
    save_all=True,
    append_images=frames[1:],
    loop=0,
    duration=int(1000 / fps),
)
print(f'\n✓ Saved {filename}  ({len(frames)} frames, {fps} fps)')

display(IPyImage(filename=filename))


In [ ]:
# Find period-3 orbits (accessible periodic orbits in this parameter regime)
print("Searching for period-3 periodic orbits...")
period = 3
cycles_p3 = cf.find_period_p_orbits(
    kappa_classical, alpha_classical, order=order, p=period,
    grid_th=41, grid_ph=81,
    seed_tol=5e-2, refine_tol=1e-10
)

if cycles_p3 and cycles_p3[0]:
    # Flatten the nested structure
    cycles_p3 = cycles_p3[0] if isinstance(cycles_p3[0], list) and len(cycles_p3[0]) == 1 else cycles_p3
    print(f"✓ Found {len(cycles_p3)} period-{period} orbits")
    
    # Display orbit details
    for i, orbit in enumerate(cycles_p3[:3]):  # Show first 3
        print(f"\n  Orbit {i+1}:")
        for j, (th, ph) in enumerate(orbit[:period]):
            print(f"    Point {j+1}: θ={th:.4f} rad, φ={ph:.4f} rad")
else:
    print(f"⚠ No period-{period} orbits found in this parameter regime")
    print("  This is normal for chaotic parameters. Quantum scars will show up in eigenstate analysis.")

In [ ]:
# Recreate the Poincaré section with periodic orbits overlaid
fig, ax = plt.subplots(figsize=(10, 8))

# Plot phase space points (same as before)
ax.scatter(phi_pts, theta_pts, c='gray', s=0.3, alpha=0.4, label='Chaotic trajectories')

# Overlay periodic orbits if they were found
if cycles_p3 and cycles_p3[0]:
    # Extract and plot each periodic orbit
    colors_orbits = plt.cm.tab10(np.linspace(0, 1, len(cycles_p3)))
    
    for orbit_idx, orbit in enumerate(cycles_p3):
        # Flatten nested structure if needed
        orbit_pts = orbit[0] if isinstance(orbit[0], list) else orbit
        
        # Extract spherical coordinates
        thetas_orbit = [th for th, ph in orbit_pts[:period]]
        phis_orbit = [ph for th, ph in orbit_pts[:period]]
        
        # Plot orbit as connected line and points
        thetas_plot = thetas_orbit + [thetas_orbit[0]]  # Close the loop
        phis_plot = phis_orbit + [phis_orbit[0]]
        
        ax.plot(phis_plot, thetas_plot, color=colors_orbits[orbit_idx], 
                linewidth=2.5, alpha=0.8, label=f'Period-{period} Orbit {orbit_idx+1}')
        ax.scatter(phis_orbit, thetas_orbit, color=colors_orbits[orbit_idx], 
                  s=100, marker='o', edgecolors='black', linewidth=1, zorder=5)

ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(0, np.pi)
ax.set_xlabel(r'$\phi$ (azimuthal angle)', fontsize=12)
ax.set_ylabel(r'$\theta$ (polar angle)', fontsize=12)
ax.set_title(f'Phase Space with Classical Periodic Orbits (κ={kappa:.2f}, p={order})', fontsize=13, fontweight='bold')

# Formatting
ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'], fontsize=10)
ax.set_yticks([0, np.pi/2, np.pi])
ax.set_yticklabels([r'$0$', r'$\pi/2$', r'$\pi$'], fontsize=10)
ax.grid(True, alpha=0.3, linestyle=':')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

print("✓ Periodic orbits overlaid on Poincaré section")
if cycles_p3 and cycles_p3[0]:
    print(f"\nPeriodic orbits visualized:")
    print(f"  - Gray dots: Chaotic trajectories filling the ergodic region")
    print(f"  - Colored lines: Stable periodic orbits (if they exist)")
    print(f"  - Number of orbits found: {len(cycles_p3)}")
else:
    print("\nNote: No periodic orbits found in this parameter regime.")
    print("This is typical for strong chaos (κ > 2). Quantum scars still exist but become fainter.")


### 5.1 Initialize Quantum Initial State and Evolve

We start with a spin-coherent state and iteratively apply the Floquet operator. This simulates the real-time evolution under stroboscopic kicks. We track:
- **Fidelity** $F(t) = |\langle \psi(0)|\psi(t)\rangle|^2$ → decays due to sensitivity to initial conditions (butterfly effect)
- **Entropy** $S(t) = -\text{Tr}(\rho \ln \rho)$ → grows as the state spreads over the Hilbert space, approaching equilibration

These metrics reveal the **signature of chaos**: information loss and delocalization. Persistent oscillations in fidelity signal the presence of scarred eigenstates (which can "revivals" periodically).


### 5.2 Quantum Fidelity Decay and Entropy Growth

We now analyze the quantum dynamics by plotting two key observables:

1. **Fidelity decay** (left): Shows how quickly the initial state loses its coherence. The oscillatory structure reveals the Floquet spectrum.
2. **Entropy growth** (right): Von Neumann entropy increases as the state spreads over accessible Hilbert space. Saturation indicates thermalization.

The interplay between fidelity and entropy reveals the chaotic vs. scarring behavior.


In [ ]:
# Define a classical periodic orbit (or use a generic equatorial orbit if none found)
# We'll use points from the Poincaré section as a proxy
test_orbit_points = []

# Create a simple equatorial orbit (approximate)
n_orbit_points = 5
for j in range(n_orbit_points):
    phi_test = -np.pi + 2 * np.pi * j / n_orbit_points
    theta_test = np.pi / 2  # Equatorial
    test_orbit_points.append((theta_test, phi_test))

print(f"✓ Defined test orbit with {n_orbit_points} points")
print(f"\nScan Floquet eigenstates for scars...")

# Compute scar overlaps for each Floquet eigenstate
scar_overlaps = cf.compute_scar_overlaps(eigvecs_sorted, I, test_orbit_points)

# Find most scarred eigenstate
top_scarred_idx = np.argsort(scar_overlaps)[::-1][:3]  # Top 3

print(f"\nTop 3 scarred eigenstates:")
for rank, idx in enumerate(top_scarred_idx, 1):
    print(f"  {rank}. Eigenstate {idx}: scar overlap = {scar_overlaps[idx]:.6f}")

In [ ]:
# Plot scar overlaps for all eigenstates
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['red' if idx in top_scarred_idx else 'steelblue' for idx in range(dim)]
ax.bar(range(dim), scar_overlaps, color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('Floquet eigenstate index', fontsize=11)
ax.set_ylabel('Scar overlap (overlap with classical orbit)', fontsize=11)
ax.set_title(f'Scar Detection: Overlap with Test Orbit (κ={kappa/(np.pi):.1f}π)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(['Top scarred', 'Other'], fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Scar visualization complete")

In [ ]:
# Get the most scarred eigenstate
best_scar_idx = top_scarred_idx[0]
psi_scarred = eigvecs_sorted[best_scar_idx]

print(f"✓ Computing Husimi Q-function for eigenstate {best_scar_idx}")
print(f"  (scar overlap = {scar_overlaps[best_scar_idx]:.6f})")

# Compute Husimi Q-function using QuTiP's built-in function
theta_grid = np.linspace(0, np.pi, 120)
phi_grid = np.linspace(-np.pi, np.pi, 120)
Q, _, _ = spin_q_function(psi_scarred, theta_grid, phi_grid)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.contourf(phi_grid, theta_grid, Q, levels=25, cmap='hot')

# Overlay classical orbit points
if test_orbit_points:
    orbit_thetas = [th for th, ph in test_orbit_points]
    orbit_phis = [ph for th, ph in test_orbit_points]
    ax.plot(orbit_phis + [orbit_phis[0]], orbit_thetas + [orbit_thetas[0]], 
            'b-', linewidth=2.5, label='Test orbit', alpha=0.8)
    ax.scatter(orbit_phis, orbit_thetas, color='cyan', s=60, marker='x', linewidth=2.5, zorder=5)

ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(0, np.pi)
ax.set_xlabel(r'$\phi$ (azimuthal angle)', fontsize=11)
ax.set_ylabel(r'$\theta$ (polar angle)', fontsize=11)
ax.set_title(f'Husimi Q-function: Scarred Eigenstate {best_scar_idx}', fontsize=12, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r'$Q(\theta,\phi) = |\langle\theta,\phi|\psi\rangle|^2$', fontsize=10)

plt.tight_layout()
plt.show()

print("\n✓ Husimi Q-function visualization complete")
print("\nInterpretation:")
print("  - Husimi Q-function projects quantum state onto phase space")
print("  - Peaks indicate likely measurement outcomes in coherent state basis")
print("  - Concentration along blue orbit line → eigenstate is SCARRED on this classical structure")


In [ ]:
# Compute FFT spectrum of fidelity
kick_times = np.arange(len(overlaps))
freqs, spectrum = cf.fft_fidelity_spectrum(fidelity, kick_times, zero_pad_factor=4)

fig, ax = plt.subplots(figsize=(10, 5))

ax.semilogy(freqs, spectrum, 'b-', linewidth=1.5)
ax.set_xlabel('Frequency (cycles per kick)', fontsize=11)
ax.set_ylabel('Normalized FFT amplitude (log scale)', fontsize=11)
ax.set_title('Fidelity Power Spectrum', fontsize=12)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0, 0.5)  # Nyquist limit

# Mark dominant frequency
if len(freqs) > 0:
    dominant_freq_idx = np.argmax(spectrum)
    dominant_freq = freqs[dominant_freq_idx]
    ax.scatter([dominant_freq], [spectrum[dominant_freq_idx]], 
               color='red', s=100, marker='*', zorder=5, label=f'Peak @ f={dominant_freq:.3f}')
    if dominant_freq > 0:
        period = 1.0 / dominant_freq
        ax.text(dominant_freq, spectrum[dominant_freq_idx] * 2, f'Period ≈ {period:.1f} kicks',
               fontsize=10, ha='left', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"✓ FFT analysis complete")
if dominant_freq > 0:
    print(f"  Dominant frequency: {dominant_freq:.6f} cycles/kick")
    print(f"  Corresponding period: {1.0/dominant_freq:.1f} kicks")

In [ ]:
# Summary statistics
print("="*70)
print(" SIMULATION SUMMARY")
print("="*70)

print(f"\nSystem:")
print(f"  Spin quantum number: I = {I}")
print(f"  Hilbert space dimension: d = {dim}")

print(f"\nKicked-Top Hamiltonian:")
print(f"  Nonlinearity order: p = {order}")
print(f"  Kick strength: κ = {kappa:.4f} ({kappa/np.pi:.2f}π)")
print(f"  Free evolution time: τ = {tau}")

print(f"\nQuantum Dynamics:")
print(f"  Number of kicks: {n_kicks}")
print(f"  Initial state: spin-coherent at (θ={theta0:.3f}, φ={phi0:.3f})")
print(f"  Final fidelity: {fidelity[-1]:.6f}")
print(f"  Entropy growth: ΔS = {entropies[-1] - entropies[0]:.6f}")

print(f"\nFloquet Analysis:")
print(f"  Number of eigenstates: {dim}")
print(f"  Mean eigenphase spacing: {np.mean(np.diff(phases_sorted)):.6f} rad")
print(f"  Top scarred eigenstate: #{best_scar_idx}, overlap = {scar_overlaps[best_scar_idx]:.6f}")

print(f"\nClassical Phase Space:")
print(f"  Poincaré section points: {len(phi_pts)}")
if cycles_p3 and cycles_p3[0]:
    print(f"  Period-3 orbits found: {len(cycles_p3)}")
else:
    print(f"  Period-3 orbits: None (parameter regime is fully chaotic)")

print("\n" + "="*70)